# Gastrulation dataset check
取得済みAnnDataの読取専用診断。annotation移植・追加前処理・学習・velocity・benchmark指標計算は行いません。

リモートで先に `python download_data.py` を実行し、このNotebookを上から順に実行してください。パス入力は不要です。`full` はscVelo取得データの呼称であり、内容は本Notebookで確認します。

In [ ]:
from pathlib import Path
import sys
import importlib.metadata
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display

candidates = []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / 'data_preparation' / '20261007'])
candidates.append(Path('/home/suzuki/Projects/scDiffusion-github/data_preparation/20261007'))
BASE = next((p.resolve() for p in candidates if (p / 'inspection_helpers.py').is_file()
             and (p / 'download_data.py').is_file()), None)
if BASE is None:
    raise FileNotFoundError('Notebookの配置ディレクトリまたはリポジトリからJupyterを起動してください。')
sys.path.insert(0, str(BASE))
from inspection_helpers import (SEED, CANDIDATES, matrices, structure, obs_report,
    id_report, compare_columns, describe_matrix, library_report, positions,
    bounded_subset, agreement, scatter, histogram)
DATA = BASE / 'data'
FIGURES = BASE / 'figures'
FIGURES.mkdir(exist_ok=True)
print('BASE:', BASE, '\nDATA:', DATA, '\nFIGURES:', FIGURES)
print({p: importlib.metadata.version(p) for p in
       ['anndata', 'scanpy', 'scvelo', 'numpy', 'scipy', 'pandas', 'matplotlib', 'h5py']})
for name in ['gastrulation_full.h5ad', '03_Gastrulation_erythroid.h5ad']:
    if not (DATA / name).is_file():
        raise FileNotFoundError(f'{DATA / name}: リモートで先にdownload_data.pyを実行してください。')
full = sc.read_h5ad(DATA / 'gastrulation_full.h5ad')
ery = sc.read_h5ad(DATA / '03_Gastrulation_erythroid.h5ad')
datasets = {'full': full, 'ery': ery}

## A. AnnData全体構造と各matrixの統計
Xと全layersを走査します。全matrixのdense化はありません。基本momentsは全要素、quantileは推定値です。`.raw` の存在とshapeも確認しますが、名前だけではraw countとは決めません。

In [ ]:
diagnostics = {}
for name, adata in datasets.items():
    print('\n===', name, '===')
    structure(adata)
    diagnostics[name] = {}
    for key, matrix in matrices(adata):
        print('Scanning:', name, key, matrix.shape, flush=True)
        diagnostics[name][key] = describe_matrix(matrix)
    display(pd.DataFrame({k: v['summary'] for k, v in diagnostics[name].items()}).T)

## B. obsの全column・候補annotation

In [ ]:
for name, adata in datasets.items():
    print('\n===', name, 'OBS ===')
    obs_report(adata.obs)

## C. cell IDの対応
ID文字列は加工しません。重複を報告し、一意対応するIDだけを以後の値比較に使います。

In [ ]:
print('full.obs_names:', full.obs_names)
print('ery.obs_names:', ery.obs_names)
cell_match, common_ids, aligned_cells = id_report(full.obs_names, ery.obs_names, 'cells')
ery_only_ids = ery.obs_names.unique().difference(full.obs_names.unique(), sort=False)
full_only_ids = full.obs_names.unique().difference(ery.obs_names.unique(), sort=False)

## D. 共通obs columnの対応
一致率は双方nonmissingのcell上で計算します。celltype/stageを正解と決めず、全共通columnを確認します。

In [ ]:
common_obs_columns = full.obs.columns.intersection(ery.obs.columns, sort=False)
obs_agreement = compare_columns(full.obs, ery.obs, aligned_cells)
for col in CANDIDATES:
    if col in obs_agreement.index:
        print('Candidate:', col)
        display(obs_agreement.loc[col])
    else:
        print(col, ': not comparable / not shared')

## E. gene IDの対応・共通var metadata

In [ ]:
gene_match, common_genes, aligned_genes = id_report(full.var_names, ery.var_names, 'genes')
var_agreement = compare_columns(full.var, ery.var, aligned_genes, crosstabs=False)

## F. preprocessing状態の診断
整数らしさ、負値、値域、library size、metadataを観察します。これだけでraw/normalized/log1pを断定しません。quantileはゼロを含む座標sample、整数らしさはnonzero sampleです。

以後の発現比較は両datasetで一意に対応するcell/geneから固定seedで最大500×500を抽出し、同じ順序で参照します。

In [ ]:
for name, entries in diagnostics.items():
    print('\n', name, 'X / all layers: preprocessing evidence')
    display(pd.DataFrame({k: v['summary'] for k, v in entries.items()}).T)
    display(pd.DataFrame({k: library_report(v['library']) for k, v in entries.items()
                          if k in ['X', 'spliced', 'unspliced']}).T)

rng = np.random.default_rng(SEED)
sampled_cells = aligned_cells.take(rng.choice(len(aligned_cells), min(500, len(aligned_cells)), replace=False))
sampled_genes = aligned_genes.take(rng.choice(len(aligned_genes), min(500, len(aligned_genes)), replace=False))
print('Matched sample shape:', len(sampled_cells), len(sampled_genes))
print('Sample cell IDs:', sampled_cells[:20].tolist())
print('Sample gene IDs:', sampled_genes[:20].tolist())
subsets = {name: {} for name in datasets}
if len(sampled_cells) and len(sampled_genes):
    for name, adata in datasets.items():
        ri = positions(adata.obs_names, sampled_cells)
        ci = positions(adata.var_names, sampled_genes)
        assert adata.obs_names.take(ri).equals(sampled_cells)
        assert adata.var_names.take(ci).equals(sampled_genes)
        for key, matrix in matrices(adata):
            if key in ['X', 'spliced', 'unspliced']:
                subsets[name][key] = bounded_subset(matrix, ri, ci)
        local = subsets[name]
        if 'X' in local and 'spliced' in local:
            print(name, 'X vs spliced')
            display(pd.Series(agreement(local['X'], local['spliced'])))
            if 'unspliced' in local:
                print(name, 'X vs spliced + unspliced')
                display(pd.Series(agreement(local['X'], local['spliced'] + local['unspliced'])))
else:
    print('No unambiguous matched cell/gene subset; paired expression diagnostics skipped.')

## G. 同一cell・geneの発現値比較
順序を揃えたsubset上の比較です。差異があっても停止しません。sampleでの完全一致は全データでの完全一致を証明しません。

In [ ]:
expression_agreement = {}
for key in ['X', 'spliced', 'unspliced']:
    if key in subsets['full'] and key in subsets['ery']:
        expression_agreement[key] = agreement(subsets['full'][key], subsets['ery'][key])
        print(key)
        display(pd.Series(expression_agreement[key]))
    else:
        print(key, ': missing matrix or no matched sample; skipped')

## H. 発現量の可視化
PNGを `figures/` に保存します。gene統計は全cellから計算済みの値を使用します。scatterは最大50,000点。対数図は正値のみを表示します。nonzeroヒストグラムはsample分布です。

In [ ]:
for name, entries in diagnostics.items():
    for key in ['X', 'spliced', 'unspliced']:
        if key not in entries:
            continue
        d = entries[key]
        for log in [False, True]:
            suffix = '_log' if log else ''
            scatter(d['gene_mean'], d['gene_var'], 'Gene mean', 'Gene variance',
                    FIGURES / f'{name}_{key}_gene_mean_vs_variance{suffix}.png', log=log)

for key in ['X', 'spliced', 'unspliced']:
    if all(key in diagnostics[name] for name in datasets) and len(aligned_genes):
        fi = positions(full.var_names, aligned_genes)
        ei = positions(ery.var_names, aligned_genes)
        assert full.var_names.take(fi).equals(ery.var_names.take(ei))
        scatter(diagnostics['full'][key]['gene_mean'][fi], diagnostics['ery'][key]['gene_mean'][ei],
                'Full gene mean (all full cells)', 'Ery gene mean (all ery cells)',
                FIGURES / f'full_vs_ery_{key}_gene_mean.png')
    if key in expression_agreement:
        scatter(subsets['full'][key], subsets['ery'][key], 'Full expression', 'Ery expression',
                FIGURES / f'matched_{key}_expression.png')
    for field, title in [('library', 'library_size'), ('nonzero_sample', 'nonzero_expression_sample')]:
        series = {name: entries[key][field] for name, entries in diagnostics.items() if key in entries}
        for clipped in [False, True]:
            suffix = '_q999' if clipped else ''
            histogram(series, f'{key} {title}', FIGURES / f'{key}_{title}{suffix}.png', clipped)

## I. 自動summary
データに対する変更・annotation採用の最終判断は行いません。

In [ ]:
for name, adata in datasets.items():
    print('\n', name.upper())
    print('shape:', adata.shape)
    print('obs columns:', adata.obs.columns.tolist())
    print('layers:', list(adata.layers.keys()))
    for key in ['X', 'spliced', 'unspliced']:
        info = diagnostics[name].get(key)
        if info is None:
            print(key, ': absent')
        else:
            s = info['summary']
            print(key, ':', {k: s[k] for k in ['storage', 'dtype', 'integer_like_fraction',
                                               'min', 'max', 'zero_fraction_finite']})
print('\nCELL MATCH'); display(pd.Series(cell_match))
print('\nGENE MATCH'); display(pd.Series(gene_match))
print('\nCOMMON OBS COLUMNS:', common_obs_columns.tolist())
for col in ['celltype', 'stage']:
    print(col, 'agreement:')
    display(obs_agreement.loc[col] if col in obs_agreement.index else 'not comparable / not shared')
print('\nEXPRESSION AGREEMENT (sample only)')
for key in ['X', 'spliced', 'unspliced']:
    print(key)
    display(pd.Series(expression_agreement[key]) if key in expression_agreement else 'not comparable / absent')
print('Figures:', FIGURES)
print('未確定: raw countの由来、前処理の正確な手順、Task I/IIの採用annotation。')
print('annotation移植・追加前処理・学習・評価指標の計算は実施していません。')